# Deciding the Optimal Split Point for Categorical Attributes

This notebook illustrates how to determine the best split for categorical attributes in Decision Trees. As seen in our lecture, we have two primary approaches for splitting a categorical attribute like **CarType**:

1. **Multi-way split**: Each distinct category becomes its own branch.
2. **Two-Way Split (Binary Split)**: Groups the categories into two subsets and evaluates how well each grouping separates the classes.

Let's use the exact example from the slide:
- **Attribute**: `CarType` with categories `{Family, Sports, Luxury}`
- **Classes**: `C1` and `C2`

In [4]:
import pandas as pd
import numpy as np
import itertools
from IPython.display import display

# Create the dataset matching the slide
data = {
    'CarType': ['Family']*5 + ['Sports']*3 + ['Luxury']*2,
    'Class': ['C1', 'C2', 'C2', 'C2', 'C2',  # Family: 1 C1, 4 C2
              'C1', 'C1', 'C2',              # Sports: 2 C1, 1 C2
              'C1', 'C2']                    # Luxury: 1 C1, 1 C2
}
df = pd.DataFrame(data)

display(df.head)

# Display the class distribution summary table
class_distribution = pd.crosstab(df['Class'], df['CarType'])
print("Data Distribution:")
display(class_distribution)

<bound method NDFrame.head of   CarType Class
0  Family    C1
1  Family    C2
2  Family    C2
3  Family    C2
4  Family    C2
5  Sports    C1
6  Sports    C1
7  Sports    C2
8  Luxury    C1
9  Luxury    C2>

Data Distribution:


CarType,Family,Luxury,Sports
Class,,,
C1,1,1,2
C2,4,1,1


## 1. Gini Impurity Function
First, we need a function to calculate the Gini impurity of a single group.
$$Gini = 1 - \sum (p_i)^2$$
Where $p_i$ is the probability of an item belonging to class $i$ in that group.

In [6]:
def gini_impurity(class_counts):
    """Calculate Gini Impurity for a list of class counts."""
    total = sum(class_counts)
    if total == 0:
        return 0.0
    probabilities = [count / total for count in class_counts]
    gini = 1.0 - sum(p**2 for p in probabilities)
    return gini

# Example: Gini for the "Family" category
family_counts = class_distribution['Family'].values
print(f"Gini for 'Family' node {list(family_counts)}: {gini_impurity(family_counts):.2f}")

Gini for 'Family' node [np.int64(1), np.int64(4)]: 0.32


## 2. Multi-way Split
In a multi-way split, we calculate the Gini impurity for each category separately, and then compute the weighted average based on the number of instances in each category.

In [ ]:
def split_weighted_gini(groups):
    """Calculate the weighted Gini impurity of a split."""
    total_instances = sum(sum(group) for group in groups)
    weighted_gini = 0.0
    
    for group in groups:
        group_total = sum(group)
        if group_total == 0:
            continue
        group_gini = gini_impurity(group)
        weight = group_total / total_instances
        weighted_gini += weight * group_gini
        
    return weighted_gini

# Groups for Multi-way split
multi_way_groups = [
    class_distribution['Family'].values,
    class_distribution['Sports'].values,
    class_distribution['Luxury'].values
]

multi_way_gini = split_weighted_gini(multi_way_groups)
print(f"Multi-way split Gini: {multi_way_gini:.3f}")

This result (`0.393`) matches the multi-way split calculation on the slide exactly!

## 3. Two-Way (Binary) Splits
For a two-way split, we must evaluate all possible partitions of the categories into two subsets. 
With 3 categories `{Family, Sports, Luxury}`, the possible non-empty binary partitions are:
1. `{Sports, Luxury}` vs `{Family}`
2. `{Family, Luxury}` vs `{Sports}`
3. `{Family, Sports}` vs `{Luxury}`

In [ ]:
categories = class_distribution.columns.tolist()

# Find all ways to split categories into 2 groups
def get_binary_splits(categories):
    n = len(categories)
    splits = []
    # We only need to go up to 2**(n-1) to avoid duplicate complementary pairs
    # e.g., Group1 vs Group2 is the same partition as Group2 vs Group1
    for i in range(1, 2**(n-1)):
        group1 = [categories[j] for j in range(n) if (i & (1 << j))]
        group2 = [cat for cat in categories if cat not in group1]
        splits.append((group1, group2))
    return splits

binary_splits = get_binary_splits(categories)

best_gini = float('inf')
best_split = None

print("Evaluating all Two-Way Splits:\n")
for group1, group2 in binary_splits:
    # Sum counts for each group
    counts1 = class_distribution[group1].sum(axis=1).values
    counts2 = class_distribution[group2].sum(axis=1).values
    
    gini = split_weighted_gini([counts1, counts2])
    
    print(f"Split: {group1} vs {group2}")
    print(f"  Gini: {gini:.3f}\n")
    
    if gini < best_gini:
        best_gini = gini
        best_split = (group1, group2)

print(f"\ud83c\udfc6 Best Two-Way Split is {best_split[0]} vs {best_split[1]} with Gini {best_gini:.3f}")

These results (`0.400` and `0.419`) also perfectly match the slide! 
The optimal two-way split groups `{Sports, Luxury}` together against `{Family}`.

## 4. Visualizing the Splits
Let's create a visual representation of how the tree partitions the data.

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

def draw_split_tree(title, root_label, branches):
    G = nx.DiGraph()
    
    G.add_node("Root", label=root_label)
    
    pos = {"Root": (0, 1)}
    labels = {"Root": root_label}
    
    n_branches = len(branches)
    xs = np.linspace(-1, 1, n_branches)
    
    for i, branch in enumerate(branches):
        node_id = f"Node_{i}"
        G.add_node(node_id, label=branch['label'])
        G.add_edge("Root", node_id, weight=branch['edge'])
        
        pos[node_id] = (xs[i], 0)
        labels[node_id] = branch['label']
        
    plt.figure(figsize=(8, 4))
    plt.title(title, fontsize=14, fontweight='bold', pad=20)
    
    # Draw nodes
    nx.draw_networkx_nodes(G, pos, node_size=3000, node_color="lightblue", edgecolors="black", node_shape="s")
    
    # Draw edges
    nx.draw_networkx_edges(G, pos, arrowsize=20, width=2)
    
    # Draw node labels
    nx.draw_networkx_labels(G, pos, labels, font_size=10, font_weight="bold")
    
    # Draw edge labels
    edge_labels = nx.get_edge_attributes(G, 'weight')
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=12, font_color='red', label_pos=0.5)
    
    plt.axis('off')
    plt.show()

# Visualize Multi-way Split
multi_branches = [
    {'edge': 'Family', 'label': f"C1: {class_distribution.loc['C1', 'Family']}\nC2: {class_distribution.loc['C2', 'Family']}"},
    {'edge': 'Sports', 'label': f"C1: {class_distribution.loc['C1', 'Sports']}\nC2: {class_distribution.loc['C2', 'Sports']}"},
    {'edge': 'Luxury', 'label': f"C1: {class_distribution.loc['C1', 'Luxury']}\nC2: {class_distribution.loc['C2', 'Luxury']}"}
]
draw_split_tree("Multi-way Split (Gini: 0.393)", "CarType", multi_branches)

# Visualize Best Two-way Split
c1_sports_luxury = class_distribution.loc['C1', 'Sports'] + class_distribution.loc['C1', 'Luxury']
c2_sports_luxury = class_distribution.loc['C2', 'Sports'] + class_distribution.loc['C2', 'Luxury']

two_way_branches = [
    {'edge': '{Sports, Luxury}', 'label': f"C1: {c1_sports_luxury}\nC2: {c2_sports_luxury}"},
    {'edge': '{Family}', 'label': f"C1: {class_distribution.loc['C1', 'Family']}\nC2: {class_distribution.loc['C2', 'Family']}"},
]
draw_split_tree("Best Two-way Split (Gini: 0.400)", "CarType", two_way_branches)

## Conclusion / Summary

- **Multi-way split** produced a Gini index of `0.393` and created 3 branches.
- **Two-way split** produced a best Gini index of `0.400` and created 2 branches.
- Although the multi-way split has a slightly better (lower) Gini impurity, many algorithms (like CART) strictly use two-way splits. Binary splits help control tree complexity and prevent overfitting by not creating too many fragmented nodes.